# Praktikum Evaluasi Model Komprehensif pada Dataset Imbalanced

**Studi kasus:** Credit Card Fraud Detection  
**Cakupan:** poin A–P dari modul (poin Q/tugas praktikum tidak dikerjakan).

Tujuan utama adalah mendeteksi fraud dengan *recall* tinggi tanpa mengorbankan *precision* secara berlebihan. Selain ROC-AUC, notebook melaporkan PR-AUC karena lebih informatif untuk kelas positif yang sangat langka.

## A–B. Studi kasus dan pemuatan dataset

Dataset ULB berisi 284.807 transaksi dengan 492 fraud. Jika file belum tersedia, sel berikut mengunduhnya dari penyimpanan publik TensorFlow.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA = Path('data/creditcard.csv')
DATA.parent.mkdir(exist_ok=True)
if not DATA.exists():
    url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
    print('Mengunduh dataset (sekitar 144 MB)...')
    urlretrieve(url, DATA)
print(f'Dataset: {DATA} ({DATA.stat().st_size / 1024**2:.1f} MB)')

## C. Import library dan setup

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = Path.cwd()
DATA = ROOT / "data" / "creditcard.csv"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)


def metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def save_cm(y_true, y_pred, title, filename):
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.savefig(FIG / filename, dpi=150); plt.show()

## D. Load dataset dan eksplorasi awal

In [ ]:
print("C-D. Load dataset dan eksplorasi")
df = pd.read_csv(DATA)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Distribusi kelas:\n", df["Class"].value_counts())
print(f"Proporsi fraud: {df['Class'].mean()*100:.4f}%")
print("Statistik Amount:\n", df["Amount"].describe())
print("Mean Amount fraud:", df.loc[df.Class == 1, "Amount"].mean())
print("Mean Amount normal:", df.loc[df.Class == 0, "Amount"].mean())

counts = df["Class"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Normal", "Fraud"], counts.values, color=["steelblue", "coral"])
axes[0].set_title("Distribusi Kelas"); axes[0].set_ylabel("Jumlah")
axes[1].pie(counts.values, labels=["Normal", "Fraud"], autopct="%1.3f%%",
            colors=["steelblue", "coral"])
axes[1].set_title("Proporsi Kelas")
plt.tight_layout(); plt.savefig(FIG / "01_distribusi_kelas.png", dpi=150); plt.show()

**Temuan awal:** akurasi berpotensi menipu karena fraud hanya sekitar 0,172%. Dataset tidak memiliki missing value.

## E. Preprocessing dan split data

In [ ]:
print("\nE. Split data (scaling sengaja belum dilakukan agar tidak leakage)")
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, np.bincount(y_train))
print("Test :", X_test.shape, np.bincount(y_test))

all_results = []

Scaling belum dilakukan pada seluruh data. `StandardScaler` ditempatkan dalam pipeline agar hanya mempelajari statistik training set/fold dan mencegah leakage.

## F. Naive baseline dan akurasi yang menipu

In [ ]:
print("\nF. Naive baseline")
y_pred_naive = np.zeros(len(y_test), dtype=int)
naive = metrics(y_test, y_pred_naive, y_pred_naive.astype(float))
all_results.append({"Strategi": "Naive Baseline", **naive})
print(naive)

Model yang selalu memilih kelas normal memperoleh akurasi hampir 100%, tetapi recall fraud = 0. Artinya seluruh transaksi fraud lolos.

## G–H. Evaluasi Logistic Regression dan confusion matrix

In [ ]:
print("\nG-H. Logistic Regression baseline")
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]
lr_result = metrics(y_test, y_pred_lr, y_prob_lr)
all_results.append({"Strategi": "LR Baseline", **lr_result})
print(lr_result)
save_cm(y_test, y_pred_lr, "Confusion Matrix – Logistic Regression", "02_cm_lr.png")

## I. Handling imbalance dengan SMOTE

In [ ]:
print("\nI. Logistic Regression + SMOTE (hanya training)")
smote_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
smote_pipe.fit(X_train, y_train)
y_pred_smote = smote_pipe.predict(X_test)
y_prob_smote = smote_pipe.predict_proba(X_test)[:, 1]
smote_result = metrics(y_test, y_pred_smote, y_prob_smote)
all_results.append({"Strategi": "LR + SMOTE", **smote_result})
print(smote_result)
save_cm(y_test, y_pred_smote, "Confusion Matrix – LR + SMOTE", "03_cm_smote.png")

SMOTE hanya dijalankan pada training set melalui `imblearn.Pipeline`. Test set tetap merepresentasikan distribusi data nyata.

## J. Stratified 5-Fold Cross-Validation

In [ ]:
print("\nJ. Stratified 5-fold cross-validation")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
cv_rows = []
cv_models = {
    "Logistic Regression": lr_pipe,
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1,
                                              class_weight="balanced_subsample")),
    ]),
}
for name, model in cv_models.items():
    scores = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
    row = {"Model": name}
    for key in scoring:
        vals = scores[f"test_{key}"]
        row[f"{key}_mean"] = vals.mean()
        row[f"{key}_std"] = vals.std(ddof=1)
    cv_rows.append(row)
    print(name, row)

# RBF-SVM pada sampel stratified karena kompleksitas kernel kuadratik tidak realistis
# untuk keseluruhan 227 ribu baris training.
X_svm, _, y_svm, _ = train_test_split(
    X_train, y_train, train_size=10000, random_state=42, stratify=y_train
)
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", probability=False, random_state=42)),
])
svm_scores = cross_validate(svm_pipe, X_svm, y_svm, cv=skf, scoring=scoring, n_jobs=4)
svm_row = {"Model": "SVM RBF (sampel stratified 10k)"}
for key in scoring:
    vals = svm_scores[f"test_{key}"]
    svm_row[f"{key}_mean"] = vals.mean()
    svm_row[f"{key}_std"] = vals.std(ddof=1)
cv_rows.append(svm_row)
print(svm_row)
pd.DataFrame(cv_rows).to_csv(OUT / "cv_models.csv", index=False)

RBF-SVM dievaluasi pada sampel stratified 10.000 baris karena kompleksitas kernel pada 227 ribu data training tidak praktis untuk notebook pembelajaran.

## K. Implementasi manual Stratified K-Fold

In [ ]:
print("\nK. Manual Stratified 5-fold")
manual_rows = []
for fold, (tr, va) in enumerate(skf.split(X_train, y_train), start=1):
    model = clone(lr_pipe)
    model.fit(X_train.iloc[tr], y_train.iloc[tr])
    pred = model.predict(X_train.iloc[va])
    prob = model.predict_proba(X_train.iloc[va])[:, 1]
    row = {"Fold": fold, **metrics(y_train.iloc[va], pred, prob)}
    manual_rows.append(row)
    print(row)
manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(OUT / "manual_cv.csv", index=False)
print("Mean ± sample std:\n", manual_df[["Accuracy", "F1", "Recall", "ROC_AUC"]].agg(["mean", "std"]))

## L. GridSearchCV dengan pipeline

In [ ]:
print("\nL. GridSearchCV")
# Validation terpisah juga dipakai pada bagian N agar threshold tidak dipilih di test set.
X_core, X_val, y_core, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
param_grid = {
    "classifier__solver": ["liblinear"],
    "classifier__penalty": ["l1", "l2"],
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
}
grid = GridSearchCV(lr_pipe, param_grid, cv=skf, scoring="f1", n_jobs=4,
                    return_train_score=False, verbose=1)
grid.fit(X_core, y_core)
print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score").to_csv(
    OUT / "grid_search_results.csv", index=False
)

y_pred_best = grid.best_estimator_.predict(X_test)
y_prob_best = grid.best_estimator_.predict_proba(X_test)[:, 1]
best_result = metrics(y_test, y_pred_best, y_prob_best)
all_results.append({"Strategi": "Best GridSearch (threshold 0.5)", **best_result})
print("Test metrics:", best_result)

## M. ROC Curve dan AUC

In [ ]:
print("\nM. ROC Curve dan AUC")
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {best_result['ROC_AUC']:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random = 0.5")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Model GridSearch"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "04_roc_curve.png", dpi=150); plt.show()